# Latency benchmark — cost of adding the reranker

Measures real per-query time for retrieval alone vs. retrieval + reranking, so the paper can state an actual number rather than claiming the mitigation is "cheap" without evidence.

Reports **median and p95**, not just mean — p95 reflects the worst-case latency users actually notice, which matters more than the average for a user-facing system.

One-time costs (index building, model loading) are measured **separately** from per-query cost — they happen once per deployment, not once per request, so mixing them in would misrepresent the real overhead.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. Works on GPU or CPU — run once on each if you want to report both, since not every deployment has a GPU.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,   # candidates passed to the reranker -- same as solution_6
    "n_queries": 100,   # timed queries; run after a warmup so cold-start cost isn't included
    "n_warmup": 5,
    "seed": 42,
}

### Load data and build the index once (index-build time is NOT part

In [ ]:
# of per-query latency -- it happens once per deployment, not once per request)
import json, random, re, time
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)
timed_qa = qa[: CONFIG["n_queries"]]
warmup_qa = qa[CONFIG["n_queries"]: CONFIG["n_queries"] + CONFIG["n_warmup"]]

print(f"Corpus: {len(corpus)} passages | timing {len(timed_qa)} queries "
      f"(+{len(warmup_qa)} warmup, excluded from the measurement)")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

t0 = time.perf_counter()
bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
bm25_build_s = time.perf_counter() - t0
print(f"BM25 index built in {bm25_build_s:.2f}s (one-time cost, not per-query).")

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Build the dense index once (one-time cost)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

bi = SentenceTransformer(CONFIG["base_encoder"])
t0 = time.perf_counter()
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")
index_build_s = time.perf_counter() - t0
print(f"Dense index built in {index_build_s:.2f}s for {len(corpus)} passages (one-time cost).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

### Load the reranker once (model loading is one-time, not per-query)

In [ ]:
from sentence_transformers import CrossEncoder

ce = CrossEncoder(CONFIG["reranker"], max_length=512, trust_remote_code=True,
                  automodel_args={"torch_dtype": torch.float32})
print(f"Reranker loaded: {CONFIG['reranker']}")

### Timed functions for each stage

In [ ]:
def timed_retrieve(query, k):
    """Hybrid retrieval for one query. Returns (candidate_ids, elapsed_seconds)."""
    t0 = time.perf_counter()
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    dense = minmax(corpus_emb @ q)
    sparse = minmax(np.asarray(bm25.get_scores(tokenize(query))))
    s = CONFIG["alpha"] * dense + (1 - CONFIG["alpha"]) * sparse
    idx = np.argsort(-s)[: k]
    elapsed = time.perf_counter() - t0
    return [corpus_ids[i] for i in idx], elapsed

def timed_rerank(query, candidate_ids):
    """Cross-encoder rerank of an already-retrieved candidate list.
    Returns (reordered_ids, elapsed_seconds)."""
    t0 = time.perf_counter()
    pairs = [(query, corpus_map[c]) for c in candidate_ids]
    scores = ce.predict(pairs, batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    elapsed = time.perf_counter() - t0
    return [candidate_ids[i] for i in order], elapsed

### Warmup (excluded from the measurement: first-call overhead --

In [ ]:
# CUDA kernel compilation, tokenizer caching -- is not representative of
# steady-state per-query cost)
print("Warming up...")
for q in warmup_qa:
    _, _ = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])
    cands, _ = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])
    _, _ = timed_rerank(q["darija_query"], cands)
print("Warmup done.")

### The timed run

In [ ]:
rows = []
for q in timed_qa:
    cands, t_retrieve = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])
    _, t_rerank = timed_rerank(q["darija_query"], cands)
    rows.append({
        "qid": q["id"],
        "retrieve_ms": t_retrieve * 1000,
        "rerank_ms": t_rerank * 1000,
        "total_ms": (t_retrieve + t_rerank) * 1000,
    })

lat = pd.DataFrame(rows)
lat.to_csv("latency_measurements.csv", index=False)
print(f"Timed {len(lat)} queries.")

### Summary statistics (the numbers to put in the paper)

In [ ]:
def stats(series):
    return {
        "mean": series.mean(), "median": series.median(),
        "p95": series.quantile(0.95), "min": series.min(), "max": series.max(),
    }

print("=" * 70)
print(f"LATENCY SUMMARY (n={len(lat)} queries, device={device})")
print("=" * 70)
for col, label in [("retrieve_ms", "Retrieval only (BM25 + dense)"),
                    ("rerank_ms", "Reranking only (bge-reranker-v2-m3)"),
                    ("total_ms", "Total (retrieval + reranking)")]:
    s = stats(lat[col])
    print(f"\n{label}:")
    print(f"  mean {s['mean']:.1f} ms | median {s['median']:.1f} ms | "
          f"p95 {s['p95']:.1f} ms | range [{s['min']:.1f}, {s['max']:.1f}] ms")

overhead_pct = (lat.rerank_ms.mean() / lat.retrieve_ms.mean()) * 100
print(f"\nReranking adds {lat.rerank_ms.mean():.1f} ms on top of "
      f"{lat.retrieve_ms.mean():.1f} ms retrieval "
      f"({overhead_pct:.0f}% relative overhead).")

print(f"""
One-time costs (not per-query, reported separately):
  BM25 index build     {bm25_build_s:.2f} s  for {len(corpus)} passages
  Dense index build    {index_build_s:.2f} s  for {len(corpus)} passages

Suggested paper sentence:
  "On a {device.upper()}, hybrid retrieval over {len(corpus)} passages took a median of
  {stats(lat['retrieve_ms'])['median']:.0f} ms per query; adding bge-reranker-v2-m3
  reranking of the top-{CONFIG['retrieve_k']} candidates added a median of
  {stats(lat['rerank_ms'])['median']:.0f} ms ({overhead_pct:.0f}% relative overhead),
  for a total of {stats(lat['total_ms'])['median']:.0f} ms per query
  (p95: {stats(lat['total_ms'])['p95']:.0f} ms)."
""")

from google.colab import files
files.download("latency_measurements.csv")